# California Mortgage Lending Analysis — Data Cleaning

This notebook cleans and prepares the California HMDA data for subsequent analysis. The cleaning process focuses on data types, missing and special values, invalid or implausible values, and categorical code handling.

Data types are determined based on the analytical meaning of each variable rather than only the type inferred when the CSV file is loaded. For example, some categorical variables are represented by numerical codes, but these codes should not be treated as true numerical measurements. Properly identifying and cleaning these variables helps prevent misleading calculations and interpretations.

## 1. Setup

In [1]:
import pandas as pd

In [2]:
file_path = "../data/raw/state_CA.csv"

In [3]:
header_df = pd.read_csv(file_path, nrows=0)

In [4]:
selected_columns = [
    # Identification
    "lei",
    "activity_year",

    # Geography
    "state_code",
    "county_code",
    "census_tract",

    # Application outcome
    "action_taken",

    # Loan characteristics
    "loan_type",
    "loan_purpose",
    "loan_amount",
    "loan_to_value_ratio",
    "loan_term",

    # Pricing
    "interest_rate",
    "rate_spread",
    "total_loan_costs",

    # Borrower financial characteristics
    "income",
    "debt_to_income_ratio",

    # Property
    "property_value",
    "occupancy_type",

    # Denial reasons
    "denial_reason-1",
    "denial_reason-2",
    "denial_reason-3",
    "denial_reason-4"
]

In [5]:
missing_columns = [
    col for col in selected_columns
    if col not in header_df.columns
]

missing_columns

[]

## 2. Load Selected Data

Load the 22 selected analytical variables from the full California HMDA dataset for cleaning and validation.

In [6]:
df = pd.read_csv(
    file_path,
    usecols=selected_columns,
    engine="python"
)

In [7]:
df.shape

(1026119, 22)

## 3. Pre-Cleaning Data Audit

Inspect the structure, data types, missing values, and representative values before making any transformations. This establishes a baseline for identifying variables that require cleaning.

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1026119 entries, 0 to 1026118
Data columns (total 22 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   activity_year         1026119 non-null  int64  
 1   lei                   1026119 non-null  str    
 2   state_code            1026119 non-null  str    
 3   county_code           1019645 non-null  float64
 4   census_tract          1017653 non-null  float64
 5   action_taken          1026119 non-null  int64  
 6   loan_type             1026119 non-null  int64  
 7   loan_purpose          1026119 non-null  int64  
 8   loan_amount           1026119 non-null  float64
 9   loan_to_value_ratio   693086 non-null   str    
 10  interest_rate         647211 non-null   str    
 11  rate_spread           488941 non-null   str    
 12  total_loan_costs      442107 non-null   str    
 13  loan_term             1005146 non-null  str    
 14  property_value        805951 non-null   str  

### 3.1 Inspect Categorical Codes and Special Values

In [9]:
df["action_taken"].value_counts(dropna=False)

action_taken
1    511270
3    180539
4    142155
6     96647
5     55626
2     36102
8      3196
7       584
Name: count, dtype: int64

In [10]:
categorical_cols = [
    "loan_type",
    "loan_purpose",
    "occupancy_type"
]

for col in categorical_cols:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False).sort_index())




loan_type
loan_type
1    843062
2    131111
3     51240
4       706
Name: count, dtype: int64

loan_purpose
loan_purpose
1     463320
2     120209
4     138135
5       1486
31    124117
32    178852
Name: count, dtype: int64

occupancy_type
occupancy_type
1    915898
2     16409
3     93812
Name: count, dtype: int64


In [11]:
denial_reason_cols = [
    "denial_reason-1",
    "denial_reason-2",
    "denial_reason-3",
    "denial_reason-4"
]

for col in denial_reason_cols:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False).sort_index())


denial_reason-1
denial_reason-1
1        68715
2         1142
3        30381
4        22205
5         5137
6         7409
7        29076
8           27
9        16153
10      840968
1111      4906
Name: count, dtype: int64

denial_reason-2
denial_reason-2
1.0      5753
2.0      1170
3.0      6747
4.0      5760
5.0      2958
6.0      3623
7.0      1534
8.0        21
9.0      7095
NaN    991458
Name: count, dtype: int64

denial_reason-3
denial_reason-3
1.0        353
2.0        112
3.0        368
4.0        948
5.0       1058
6.0        801
7.0        311
8.0         23
9.0       1967
NaN    1020178
Name: count, dtype: int64

denial_reason-4
denial_reason-4
1.0         27
2.0         18
3.0         28
4.0         23
5.0         55
6.0         82
7.0         35
8.0          3
9.0        771
NaN    1025077
Name: count, dtype: int64


In [12]:
interest_rate_numeric = pd.to_numeric(
    df["interest_rate"],
    errors="coerce"
)

In [13]:
df.loc[
    df["interest_rate"].notna() & interest_rate_numeric.isna(),
    "interest_rate"
].value_counts()

interest_rate
Exempt    4814
Name: count, dtype: int64

In [14]:
df["interest_rate"].isna().sum()

np.int64(378908)

In [15]:
interest_rate_numeric.isna().sum()

np.int64(383722)

In [16]:
df["interest_rate"] = pd.to_numeric(
    df["interest_rate"],
    errors="coerce"
)

In [17]:
df["interest_rate"].dtype

dtype('float64')

In [18]:
df["interest_rate"].isna().sum()

np.int64(383722)

In [19]:
numeric_cols = [
    "loan_to_value_ratio",
    "rate_spread",
    "total_loan_costs",
    "loan_term",
    "property_value"
]

In [20]:
for col in numeric_cols:
    numeric_version = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    invalid_values = df.loc[
        df[col].notna() & numeric_version.isna(),
        col
    ].value_counts()

    print(f"\n{col}")
    print(invalid_values)


loan_to_value_ratio
loan_to_value_ratio
Exempt    4637
Name: count, dtype: int64

rate_spread
rate_spread
Exempt    5305
Name: count, dtype: int64

total_loan_costs
total_loan_costs
Exempt    6216
Name: count, dtype: int64

loan_term
loan_term
Exempt    4936
Name: count, dtype: int64

property_value
property_value
Exempt    4766
Name: count, dtype: int64


In [21]:
for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [22]:
for col in numeric_cols:
    print(col, df[col].dtype)

loan_to_value_ratio float64
rate_spread float64
total_loan_costs float64
loan_term float64
property_value float64


In [23]:
dti_numeric = pd.to_numeric(
    df["debt_to_income_ratio"],
    errors="coerce"
)

In [24]:
invalid_values = df.loc[
    df["debt_to_income_ratio"].notna() & dti_numeric.isna(),
    "debt_to_income_ratio"
].value_counts()

print(invalid_values)

debt_to_income_ratio
30%-<36%    77079
50%-60%     73537
20%-<30%    63218
>60%        59188
<20%        28595
Exempt       4637
Name: count, dtype: int64


In [25]:
df["dti_numeric"] = pd.to_numeric(
    df["debt_to_income_ratio"],
    errors="coerce"
)

In [26]:
df["debt_to_income_ratio"].unique()

<StringArray>
[      '41',       '48',       '46',       '40',        nan, '30%-<36%',
       '39',     '<20%', '20%-<30%',       '47',       '38',       '45',
       '43',       '49',  '50%-60%',       '37',       '36',       '44',
       '42',     '>60%',   'Exempt']
Length: 21, dtype: str

In [27]:
df["dti_numeric"].dtype

dtype('float64')

In [28]:
df["dti_category"] = pd.NA

In [29]:
df.loc[
    df["debt_to_income_ratio"].isin([
        "<20%",
        "20%-<30%",
        "30%-<36%",
        "50%-60%",
        ">60%"
    ]),
    "dti_category"
] = df["debt_to_income_ratio"]

In [30]:
df.loc[
    df["dti_numeric"].between(36, 49, inclusive="both"),
    "dti_category"
] = "36%-<50%"

In [31]:
df["dti_category"].value_counts(dropna=False)

dti_category
<NA>        366711
36%-<50%    357791
30%-<36%     77079
50%-60%      73537
20%-<30%     63218
>60%         59188
<20%         28595
Name: count, dtype: int64

In [32]:
df["dti_category"].value_counts(dropna=False).sum()

np.int64(1026119)

In [33]:
df["dti_numeric"].between(
    36, 49,
    inclusive="both"
).sum()

np.int64(357791)

In [34]:
original_dti_missing = df["debt_to_income_ratio"].isna().sum()

In [35]:
dti_exempt = (
    df["debt_to_income_ratio"] == "Exempt"
).sum()

In [36]:
original_dti_missing + dti_exempt

np.int64(366711)

## 4. Clean Categorical Variables

Convert HMDA categorical codes into descriptive labels so that coded
identifiers are not mistakenly interpreted as numerical measurements.

### 4.1 Loan Type

In [37]:
loan_type_labels = {
    1: "Conventional",
    2: "FHA",
    3: "VA",
    4: "USDA/RHS"
}

In [38]:
df["loan_type_label"] = (
    df["loan_type"].map(loan_type_labels)
)

In [39]:
df[
    ["loan_type", "loan_type_label"]
].drop_duplicates().sort_values("loan_type")

,loan_type,loan_type_label
0,1,Conventional
13,2,FHA
6,3,VA
1,4,USDA/RHS


In [40]:
df.loc[
    df["loan_type"].notna() & df["loan_type_label"].isna(),
    "loan_type"
].value_counts()

Series([], Name: count, dtype: int64)

### 4.2 Loan Purpose

In [41]:
loan_purpose_labels = {
    1: "Home Purchase",
    2: "Home Improvement",
    4: "Other Purpose",
    5: "Not Applicable",
    31: "Refinancing",
    32: "Cash-out Refinancing"
}

In [42]:
df["loan_purpose_label"] = (
    df["loan_purpose"].map(loan_purpose_labels)
)

In [43]:
df[
    ["loan_purpose", "loan_purpose_label"]
].drop_duplicates().sort_values("loan_purpose")

,loan_purpose,loan_purpose_label
0,1,Home Purchase
92,2,Home Improvement
38,4,Other Purpose
17165,5,Not Applicable
5,31,Refinancing
3,32,Cash-out Refinancing


In [44]:
df.loc[
    df["loan_purpose"].notna() & df["loan_purpose_label"].isna(),
    "loan_purpose"
].value_counts()

Series([], Name: count, dtype: int64)

### 4.3 Action Taken

In [45]:
action_labels = {
    1: "Loan Originated",
    2: "Approved, Not Accepted",
    3: "Denied",
    4: "Withdrawn",
    5: "Closed for Incompleteness",
    6: "Purchased Loan",
    7: "Preapproval Denied",
    8: "Preapproval Approved, Not Accepted"
}

In [46]:
df["action_label"] = (
    df["action_taken"].map(action_labels)
)

In [47]:
df[
    ["action_taken", "action_label"]
].drop_duplicates().sort_values("action_taken")

,action_taken,action_label
0,1,Loan Originated
55,2,"Approved, Not Accepted"
43,3,Denied
4,4,Withdrawn
42,5,Closed for Incompleteness
18,6,Purchased Loan
129,7,Preapproval Denied
58,8,"Preapproval Approved, Not Accepted"


In [48]:
df.loc[
    df["action_taken"].notna() & df["action_label"].isna(),
    "action_taken"
].value_counts()

Series([], Name: count, dtype: int64)

### 4.4 Occupancy Type

In [49]:
occupancy_labels = {
    1: "Principal Residence",
    2: "Second Residence",
    3: "Investment Property"
}

In [50]:
df["occupancy_label"] = (
    df["occupancy_type"].map(occupancy_labels)
)

In [51]:
df[
    ["occupancy_type", "occupancy_label"]
].drop_duplicates().sort_values("occupancy_type")

,occupancy_type,occupancy_label
0,1,Principal Residence
7,2,Second Residence
12,3,Investment Property


In [52]:
df.loc[
    df["occupancy_type"].notna() & df["occupancy_label"].isna(),
    "occupancy_type"
].value_counts()

Series([], Name: count, dtype: int64)

### 4.5 Denial Reasons

Map HMDA denial-reason codes to descriptive labels while preserving missing values for unreported additional denial reasons.

In [53]:
denial_reason_cols = [
    "denial_reason-1",
    "denial_reason-2",
    "denial_reason-3",
    "denial_reason-4"
]

In [54]:
for col in denial_reason_cols:
    print(col)
    print(
        df[col]
        .dropna()
        .value_counts()
        .sort_index()
    )
    print()

denial_reason-1
denial_reason-1
1        68715
2         1142
3        30381
4        22205
5         5137
6         7409
7        29076
8           27
9        16153
10      840968
1111      4906
Name: count, dtype: int64

denial_reason-2
denial_reason-2
1.0    5753
2.0    1170
3.0    6747
4.0    5760
5.0    2958
6.0    3623
7.0    1534
8.0      21
9.0    7095
Name: count, dtype: int64

denial_reason-3
denial_reason-3
1.0     353
2.0     112
3.0     368
4.0     948
5.0    1058
6.0     801
7.0     311
8.0      23
9.0    1967
Name: count, dtype: int64

denial_reason-4
denial_reason-4
1.0     27
2.0     18
3.0     28
4.0     23
5.0     55
6.0     82
7.0     35
8.0      3
9.0    771
Name: count, dtype: int64



In [55]:
denial_reason_labels = {
    1: "Debt-to-Income Ratio",
    2: "Employment History",
    3: "Credit History",
    4: "Collateral",
    5: "Insufficient Cash",
    6: "Unverifiable Information",
    7: "Credit Application Incomplete",
    8: "Mortgage Insurance Denied",
    9: "Other",
    10: "Not Applicable",
    1111: "Exempt"
}

In [56]:
for col in denial_reason_cols:
    label_col = col.replace("-", "_") + "_label"
    print(label_col)

denial_reason_1_label
denial_reason_2_label
denial_reason_3_label
denial_reason_4_label


In [57]:
for col in denial_reason_cols:
    label_col = col.replace("-", "_") + "_label"

    df[label_col] = df[col].map(denial_reason_labels)

In [58]:
df[
    ["denial_reason-1", "denial_reason_1_label"]
].drop_duplicates().sort_values("denial_reason-1")

,denial_reason-1,denial_reason_1_label
232,1,Debt-to-Income Ratio
1014,2,Employment History
53,3,Credit History
60,4,Collateral
529,5,Insufficient Cash
52,6,Unverifiable Information
414,7,Credit Application Incomplete
74193,8,Mortgage Insurance Denied
43,9,Other
0,10,Not Applicable


In [59]:
for col in denial_reason_cols:
    label_col = col.replace("-", "_") + "_label"

    unmapped = df.loc[
        df[col].notna() & df[label_col].isna(),
        col
    ].value_counts()

    print(f"\n{col}")
    print(unmapped)


denial_reason-1
Series([], Name: count, dtype: int64)

denial_reason-2
Series([], Name: count, dtype: int64)

denial_reason-3
Series([], Name: count, dtype: int64)

denial_reason-4
Series([], Name: count, dtype: int64)


## 5. Validate Numerical Variables

Evaluate the distributions and ranges of numerical variables to identify
missing, invalid, or implausible values before downstream analysis.

In [60]:
numeric_cols = [
    "loan_amount",
    "loan_to_value_ratio",
    "interest_rate",
    "rate_spread",
    "total_loan_costs",
    "loan_term",
    "property_value",
    "income"
]

df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
loan_amount,1026119.0,5.074460e+05,3.389703e+06,5000.000,155000.000,375000.000,625000.000,2.650005e+09
loan_to_value_ratio,688449.0,1.601824e+04,1.318129e+07,0.006,53.660,72.000,83.200,1.093680e+10
interest_rate,642397.0,7.284620e+00,1.812113e+00,0.000,6.250,6.875,7.990,2.253900e+01
rate_spread,483636.0,7.225386e-01,2.643152e+00,-600.000,-0.152,0.369,1.242,3.500000e+02
total_loan_costs,435891.0,1.031302e+04,1.049452e+04,0.000,4709.530,8182.000,14329.990,4.145353e+06
loan_term,1000210.0,3.320390e+02,7.150135e+01,1.000,360.000,360.000,360.000,7.070000e+02
property_value,801185.0,1.040322e+06,5.711054e+06,5000.000,515000.000,745000.000,1105000.000,2.147484e+09
income,885162.0,1.441520e+03,1.063446e+06,-18010.000,95.000,149.000,239.000,1.000000e+09


### 5.1 Loan-to-Value Ratio

In [61]:
df["loan_to_value_ratio"].nlargest(20)

348549    1.093680e+10
218076    4.312593e+07
941064    1.022727e+05
938282    7.340614e+04
934793    7.250480e+04
952400    7.114047e+04
949226    6.830601e+04
943086    6.366882e+04
938040    6.258487e+04
865615    6.214150e+04
939918    6.120100e+04
946310    5.557000e+04
939576    5.400412e+04
812845    4.566667e+04
949940    4.041425e+04
946545    3.783862e+04
951386    3.000000e+04
941187    1.882033e+04
934622    1.785714e+04
937962    1.623436e+04
Name: loan_to_value_ratio, dtype: float64

In [62]:
df["loan_to_value_ratio"].describe(
    percentiles=[0.90, 0.95, 0.99, 0.999]
)

count    6.884490e+05
mean     1.601824e+04
std      1.318129e+07
min      6.000000e-03
90%      9.650000e+01
95%      9.835960e+01
99%      1.016900e+02
99.9%    1.391643e+02
max      1.093680e+10
Name: loan_to_value_ratio, dtype: float64

In [63]:
extreme_ltv = df.nlargest(
    20,
    "loan_to_value_ratio"
)

extreme_ltv[
    [
        "loan_amount",
        "property_value",
        "loan_to_value_ratio"
    ]
]

,loan_amount,property_value,loan_to_value_ratio
348549,1095000.0,2505000.0,1.093680e+10
218076,585000.0,NaN,4.312593e+07
941064,225000.0,5000.0,1.022727e+05
938282,55000.0,5000.0,7.340614e+04
934793,65000.0,5000.0,7.250480e+04
952400,205000.0,5000.0,7.114047e+04
949226,255000.0,5000.0,6.830601e+04
943086,105000.0,5000.0,6.366882e+04
938040,105000.0,5000.0,6.258487e+04
865615,55000.0,5000.0,6.214150e+04


In [64]:
for threshold in [100, 150, 200, 500, 1000]:
    count = (df["loan_to_value_ratio"] > threshold).sum()
    pct = count / df["loan_to_value_ratio"].notna().sum() * 100

    print(
        threshold,
        count,
        round(pct, 4)
    )

100 9702 1.4093
150 524 0.0761
200 283 0.0411
500 157 0.0228
1000 46 0.0067


In [65]:
df["ltv_extreme_flag"] = pd.NA

df.loc[
    df["loan_to_value_ratio"].notna(),
    "ltv_extreme_flag"
] = df.loc[
    df["loan_to_value_ratio"].notna(),
    "loan_to_value_ratio"
] > 200

In [66]:
df["ltv_extreme_flag"] = df["ltv_extreme_flag"].astype("boolean")

In [67]:
df["ltv_extreme_flag"].value_counts(dropna=False)

ltv_extreme_flag
False    688166
<NA>     337670
True        283
Name: count, dtype: Int64

### 5.2 Interest Rate

In [68]:
df["interest_rate"].describe(
    percentiles=[0.001, 0.01, 0.05, 0.95, 0.99, 0.999]
)

count    642397.000000
mean          7.284620
std           1.812113
min           0.000000
0.1%          0.000000
1%            2.750000
5%            5.375000
95%          10.750000
99%          13.240000
99.9%        14.990000
max          22.539000
Name: interest_rate, dtype: float64

In [69]:
df["interest_rate"].nsmallest(20)

55942    0.0
55943    0.0
56281    0.0
57030    0.0
57115    0.0
57149    0.0
57782    0.0
57922    0.0
58056    0.0
58057    0.0
58061    0.0
58063    0.0
58068    0.0
58078    0.0
58079    0.0
58175    0.0
58190    0.0
58205    0.0
58273    0.0
58278    0.0
Name: interest_rate, dtype: float64

In [70]:
df["interest_rate"].nlargest(20)

598318    22.539
269929    17.250
252649    17.125
269054    17.125
269214    17.125
269215    17.125
269435    17.125
269553    17.125
270058    17.125
260529    17.000
269044    16.875
269288    16.875
270454    16.875
246249    16.750
269018    16.750
269079    16.750
269316    16.750
270772    16.750
269303    16.625
251328    16.500
Name: interest_rate, dtype: float64

In [71]:
zero_rate_count = (df["interest_rate"] == 0).sum()

zero_rate_pct = (
    zero_rate_count
    / df["interest_rate"].notna().sum()
    * 100
)

zero_rate_count, zero_rate_pct

(np.int64(1705), np.float64(0.26541219837577074))

In [72]:
df.loc[
    df["interest_rate"] == 0,
    "action_label"
].value_counts(dropna=False)

action_label
Loan Originated                       1525
Approved, Not Accepted                  88
Purchased Loan                          81
Preapproval Approved, Not Accepted      11
Name: count, dtype: int64

### 5.3 Rate Spread

In [73]:
df["rate_spread"].describe(
    percentiles=[0.001, 0.01, 0.05, 0.95, 0.99, 0.999]
)

count    483636.000000
mean          0.722539
std           2.643152
min        -600.000000
0.1%         -6.846365
1%           -5.215650
5%           -1.059625
95%           3.887000
99%           6.261300
99.9%         7.940000
max         350.000000
Name: rate_spread, dtype: float64

In [74]:
df["rate_spread"].nsmallest(20)

665371   -600.000
287167     -8.698
474443     -8.280
549668     -8.280
549821     -8.220
989884     -8.220
992355     -8.210
989865     -8.170
989866     -8.170
993372     -8.170
549587     -8.160
989878     -8.150
989851     -8.140
993361     -8.130
989879     -8.100
7328       -7.990
7337       -7.910
7339       -7.910
7340       -7.910
474448     -7.910
Name: rate_spread, dtype: float64

In [75]:
df["rate_spread"].nlargest(20)

970443    350.0
970434    334.0
970442    325.0
970444    325.0
970459    309.0
970410    275.0
970460    264.0
970458    255.0
970441    250.0
970415    208.0
970417    207.0
970451    203.0
970456    189.0
970455    185.0
970454    165.0
970428    159.0
970418    158.0
970420    158.0
970422    158.0
970423    158.0
Name: rate_spread, dtype: float64

In [76]:
extreme_rate_spread = pd.concat([
    df.nsmallest(5, "rate_spread"),
    df.nlargest(20, "rate_spread")
])

extreme_rate_spread[
    [
        "rate_spread",
        "interest_rate",
        "action_label",
        "loan_type_label",
        "loan_purpose_label",
        "loan_amount"
    ]
]

,rate_spread,interest_rate,action_label,loan_type_label,loan_purpose_label,loan_amount
665371,-600.000,7.000,Loan Originated,Conventional,Home Improvement,105000.0
287167,-8.698,0.000,Loan Originated,Conventional,Home Purchase,245000.0
474443,-8.280,0.000,Loan Originated,Conventional,Home Improvement,25000.0
549668,-8.280,8.500,Loan Originated,Conventional,Home Improvement,115000.0
549821,-8.220,8.500,Loan Originated,Conventional,Home Improvement,125000.0
970443,350.000,8.172,Loan Originated,Conventional,Other Purpose,42005000.0
970434,334.000,7.450,Loan Originated,Conventional,Other Purpose,5185000.0
970442,325.000,7.858,Loan Originated,Conventional,Refinancing,65185000.0
970444,325.000,7.858,Loan Originated,Conventional,Refinancing,81555000.0
970459,309.000,3.710,Loan Originated,Conventional,Refinancing,5095000.0


In [77]:
df.loc[
    df["rate_spread"] > 100,
    ["lei", "rate_spread"]
].sort_values("rate_spread", ascending=False)

,lei,rate_spread
970443,549300GNIV169ZIHU012,350.0
970434,549300GNIV169ZIHU012,334.0
970444,549300GNIV169ZIHU012,325.0
970442,549300GNIV169ZIHU012,325.0
970459,549300GNIV169ZIHU012,309.0
970410,549300GNIV169ZIHU012,275.0
970460,549300GNIV169ZIHU012,264.0
970458,549300GNIV169ZIHU012,255.0
970441,549300GNIV169ZIHU012,250.0
970415,549300GNIV169ZIHU012,208.0


In [78]:
df.loc[
    df["rate_spread"] > 100,
    "lei"
].value_counts()

lei
549300GNIV169ZIHU012    45
549300LNJJYJMOPHAH96     1
Name: count, dtype: int64

In [79]:
extreme_spread_mask = (
    (df["rate_spread"] < -10) |
    (df["rate_spread"] > 100)
)

df.loc[
    extreme_spread_mask,
    ["lei", "rate_spread", "interest_rate", "action_label"]
].sort_values("rate_spread")

,lei,rate_spread,interest_rate,action_label
665371,54930057XF33SONJFP81,-600.0,7.000,Loan Originated
970414,549300GNIV169ZIHU012,101.0,5.760,Loan Originated
970412,549300GNIV169ZIHU012,126.0,5.660,Loan Originated
970411,549300GNIV169ZIHU012,126.0,5.950,Loan Originated
970421,549300GNIV169ZIHU012,126.0,5.610,Loan Originated
970435,549300GNIV169ZIHU012,128.0,4.600,Loan Originated
69988,549300LNJJYJMOPHAH96,130.0,8.250,Loan Originated
970457,549300GNIV169ZIHU012,130.0,4.250,Loan Originated
970429,549300GNIV169ZIHU012,134.0,5.380,Loan Originated
970431,549300GNIV169ZIHU012,135.0,4.960,Loan Originated


In [80]:
extreme_spread_mask.sum()

np.int64(47)

In [81]:
df["rate_spread_extreme_flag"] = pd.NA

df.loc[
    df["rate_spread"].notna(),
    "rate_spread_extreme_flag"
] = (
    (df.loc[
        df["rate_spread"].notna(),
        "rate_spread"
    ] < -10)
    |
    (df.loc[
        df["rate_spread"].notna(),
        "rate_spread"
    ] > 100)
)

df["rate_spread_extreme_flag"] = (
    df["rate_spread_extreme_flag"].astype("boolean")
)

In [82]:
df["rate_spread_extreme_flag"].value_counts(dropna=False)

rate_spread_extreme_flag
<NA>     542483
False    483589
True         47
Name: count, dtype: Int64

### 5.4 Total Loan Costs

In [83]:
df["total_loan_costs"].describe(percentiles=[0.001, 0.01, 0.05, 0.95, 0.99, 0.999])

count    4.358910e+05
mean     1.031302e+04
std      1.049452e+04
min      0.000000e+00
0.1%     0.000000e+00
1%       0.000000e+00
5%       0.000000e+00
95%      2.560492e+04
99%      3.686259e+04
99.9%    6.082339e+04
max      4.145353e+06
Name: total_loan_costs, dtype: float64

In [84]:
df["total_loan_costs"].nsmallest(20)

547      0.0
1170     0.0
1404     0.0
1702     0.0
3035     0.0
3904     0.0
5829     0.0
5901     0.0
6000     0.0
6154     0.0
6174     0.0
6184     0.0
6436     0.0
7011     0.0
7102     0.0
8579     0.0
9284     0.0
9312     0.0
9563     0.0
10387    0.0
Name: total_loan_costs, dtype: float64

In [85]:
df["total_loan_costs"].nlargest(20)

574407    4145352.52
991834     808635.00
776425     440004.86
803172     315686.59
854950     269995.17
107198     239771.50
900671     224109.60
464872     205523.90
858476     181958.40
849033     176756.70
810951     172277.00
622063     170939.78
900670     169965.31
108161     163182.48
471618     162962.50
596860     155850.07
171020     155512.74
835920     153640.07
857106     148777.66
471716     142445.00
Name: total_loan_costs, dtype: float64

In [86]:
zero_cost_count = (df["total_loan_costs"] == 0).sum()

zero_cost_pct = (
    zero_cost_count
    / df["total_loan_costs"].notna().sum()
    * 100
)

zero_cost_count, zero_cost_pct

(np.int64(24655), np.float64(5.656230571404319))

In [87]:
df.loc[
    df["total_loan_costs"] == 0,
    "action_label"
].value_counts(dropna=False)

action_label
Loan Originated    23249
Purchased Loan      1406
Name: count, dtype: int64

In [88]:
df.nlargest(20, "total_loan_costs")[
    [
        "total_loan_costs",
        "loan_amount",
        "interest_rate",
        "action_label",
        "loan_type_label",
        "loan_purpose_label",
        "lei"
    ]
]

,total_loan_costs,loan_amount,interest_rate,action_label,loan_type_label,loan_purpose_label,lei
574407,4145352.52,685000.0,5.750,Loan Originated,Conventional,Home Purchase,549300PX0B48R1TGUV89
991834,808635.00,325000.0,6.625,Loan Originated,Conventional,Home Purchase,8WH0EE09O9V05QJZ3V89
776425,440004.86,835000.0,6.750,Purchased Loan,FHA,Home Purchase,54930021WPEXNHYZUL09
803172,315686.59,245000.0,7.250,Purchased Loan,Conventional,Home Purchase,549300FNXYY540N23N64
854950,269995.17,485000.0,5.990,Purchased Loan,FHA,Home Purchase,549300LYRWPSYPK6S325
107198,239771.50,6005000.0,7.990,Loan Originated,Conventional,Home Purchase,549300A68YW07V5R5G22
900671,224109.60,18005000.0,8.625,Loan Originated,Conventional,Home Purchase,7H6GLXDRUGQFU57RNE97
464872,205523.90,215000.0,5.875,Purchased Loan,Conventional,Refinancing,549300AQ3T62GXDU7D76
858476,181958.40,725000.0,6.125,Purchased Loan,FHA,Home Purchase,549300LYRWPSYPK6S325
849033,176756.70,4355000.0,6.125,Loan Originated,VA,Cash-out Refinancing,549300LYRWPSYPK6S325


In [89]:
df["loan_cost_pct"] = (
    df["total_loan_costs"]
    / df["loan_amount"]
    * 100
)

In [90]:
df["loan_cost_pct"].describe(
    percentiles=[0.90, 0.95, 0.99, 0.999]
)

count    435891.000000
mean          2.193252
std           2.080170
min           0.000000
90%           4.579909
95%           5.290196
99%           6.188659
99.9%         7.699640
max         605.160952
Name: loan_cost_pct, dtype: float64

In [91]:
df["loan_cost_pct"].nlargest(20)

574407    605.160952
901649    269.132600
991834    248.810769
901643    191.965000
197041    189.971400
561695    173.038000
568511    159.076600
803172    128.851669
568071    113.273000
567927    110.068000
901641     99.767200
464872     95.592512
291413     93.150000
197134     78.694933
574416     70.702892
197374     68.224044
295585     64.570034
767413     63.970000
901637     62.600000
897277     58.860000
Name: loan_cost_pct, dtype: float64

In [92]:
for threshold in [10, 25, 50, 100]:
    count = (df["loan_cost_pct"] > threshold).sum()
    pct = count / df["loan_cost_pct"].notna().sum() * 100

    print(threshold, count, round(pct, 4))

10 107 0.0245
25 36 0.0083
50 26 0.006
100 10 0.0023


In [93]:
high_cost_mask = df["loan_cost_pct"] > 25

df.loc[
    high_cost_mask,
    [
        "loan_cost_pct",
        "total_loan_costs",
        "loan_amount",
        "interest_rate",
        "action_label",
        "loan_type_label",
        "loan_purpose_label",
        "lei"
    ]
].sort_values("loan_cost_pct", ascending=False)

,loan_cost_pct,total_loan_costs,loan_amount,interest_rate,action_label,loan_type_label,loan_purpose_label,lei
574407,605.160952,4145352.52,685000.0,5.750,Loan Originated,Conventional,Home Purchase,549300PX0B48R1TGUV89
901649,269.132600,13456.63,5000.0,3.750,Purchased Loan,Conventional,Not Applicable,7H6GLXDRUGQFU57RNE97
991834,248.810769,808635.00,325000.0,6.625,Loan Originated,Conventional,Home Purchase,8WH0EE09O9V05QJZ3V89
901643,191.965000,9598.25,5000.0,2.000,Purchased Loan,Conventional,Refinancing,7H6GLXDRUGQFU57RNE97
197041,189.971400,9498.57,5000.0,6.000,Purchased Loan,VA,Home Purchase,549300HW662MN1WU8550
561695,173.038000,8651.90,5000.0,6.875,Purchased Loan,Conventional,Home Purchase,254900HA4DQWAE0W3342
568511,159.076600,7953.83,5000.0,7.625,Purchased Loan,Conventional,Home Purchase,254900HA4DQWAE0W3342
803172,128.851669,315686.59,245000.0,7.250,Purchased Loan,Conventional,Home Purchase,549300FNXYY540N23N64
568071,113.273000,5663.65,5000.0,6.500,Purchased Loan,Conventional,Home Purchase,254900HA4DQWAE0W3342
567927,110.068000,5503.40,5000.0,5.875,Purchased Loan,Conventional,Home Purchase,254900HA4DQWAE0W3342


In [94]:
df.loc[
    high_cost_mask,
    "lei"
].value_counts()

lei
7H6GLXDRUGQFU57RNE97    13
549300HW662MN1WU8550     4
254900HA4DQWAE0W3342     4
549300LYRWPSYPK6S325     4
549300PX0B48R1TGUV89     2
254900ZFWS2106HWPH46     1
JJKC32MCHWDI71265Z06     1
5493006MW6O2CE88BD43     1
549300AQ3T62GXDU7D76     1
593C3GZG957YOJPS2Z63     1
54930021WPEXNHYZUL09     1
549300FNXYY540N23N64     1
KB1H1DSPRFMYMCUFXT09     1
8WH0EE09O9V05QJZ3V89     1
Name: count, dtype: int64

In [95]:
df.loc[
    high_cost_mask,
    "action_label"
].value_counts(dropna=False)

action_label
Purchased Loan     32
Loan Originated     4
Name: count, dtype: int64

In [96]:
(df.loc[high_cost_mask, "loan_amount"] == 5000).sum()

np.int64(20)

In [97]:
df["loan_cost_extreme_flag"] = pd.NA

df.loc[
    df["loan_cost_pct"].notna(),
    "loan_cost_extreme_flag"
] = (
    df.loc[
        df["loan_cost_pct"].notna(),
        "loan_cost_pct"
    ] > 25
)

df["loan_cost_extreme_flag"] = (
    df["loan_cost_extreme_flag"].astype("boolean")
)

In [98]:
df["loan_cost_extreme_flag"].value_counts(dropna=False)

loan_cost_extreme_flag
<NA>     590228
False    435855
True         36
Name: count, dtype: Int64

### 5.5 Loan Term

In [99]:
df["loan_term"].describe(
    percentiles=[0.001, 0.01, 0.05, 0.95, 0.99, 0.999]
)

count    1.000210e+06
mean     3.320390e+02
std      7.150135e+01
min      1.000000e+00
0.1%     9.000000e+00
1%       1.200000e+01
5%       1.800000e+02
95%      3.600000e+02
99%      4.800000e+02
99.9%    4.800000e+02
max      7.070000e+02
Name: loan_term, dtype: float64

In [100]:
df["loan_term"].nsmallest(20)

69965     1.0
288856    1.0
527095    1.0
535888    1.0
421918    2.0
9284      3.0
120448    3.0
218575    3.0
314300    3.0
404305    3.0
404537    3.0
588043    3.0
666222    3.0
666646    3.0
768256    3.0
768259    3.0
768260    3.0
904998    3.0
904999    3.0
73269     4.0
Name: loan_term, dtype: float64

In [101]:
df["loan_term"].nlargest(20)

474717    707.0
474521    685.0
474524    671.0
474918    665.0
474692    654.0
474519    650.0
474846    649.0
474530    648.0
474802    642.0
474699    635.0
474609    629.0
474690    628.0
474807    625.0
474808    611.0
474806    608.0
474941    606.0
474940    600.0
474867    591.0
474718    586.0
474564    582.0
Name: loan_term, dtype: float64

In [102]:
long_term_mask = df["loan_term"] > 480

long_term_mask.sum()

np.int64(82)

In [103]:
long_term_pct = (
    long_term_mask.sum()
    / df["loan_term"].notna().sum()
    * 100
)

long_term_pct

np.float64(0.008198278361544076)

In [104]:
df.loc[
    long_term_mask,
    "lei"
].value_counts().head(10)

lei
337KMNHEWWWR6B7Q7W10    73
593C3GZG957YOJPS2Z63     8
KB1H1DSPRFMYMCUFXT09     1
Name: count, dtype: int64

In [105]:
short_term_mask = df["loan_term"] < 12

short_term_mask.sum()

np.int64(1169)

In [106]:
df["loan_term_extreme_flag"] = pd.NA

df.loc[
    df["loan_term"].notna(),
    "loan_term_extreme_flag"
] = (
    df.loc[
        df["loan_term"].notna(),
        "loan_term"
    ] > 480
)

df["loan_term_extreme_flag"] = (
    df["loan_term_extreme_flag"].astype("boolean")
)

In [107]:
df["loan_term_extreme_flag"].value_counts(dropna=False)

loan_term_extreme_flag
False    1000128
<NA>       25909
True          82
Name: count, dtype: Int64

### 5.6 Property Value

In [108]:
df["property_value"].describe(
    percentiles=[0.001, 0.01, 0.05, 0.95, 0.99, 0.999]
)

count    8.011850e+05
mean     1.040322e+06
std      5.711054e+06
min      5.000000e+03
0.1%     6.500000e+04
1%       1.750000e+05
5%       3.150000e+05
95%      2.385000e+06
99%      4.605000e+06
99.9%    1.680500e+07
max      2.147484e+09
Name: property_value, dtype: float64

In [109]:
df["property_value"].nsmallest(20)

17279     5000.0
35436     5000.0
69961     5000.0
118087    5000.0
218778    5000.0
247198    5000.0
275794    5000.0
276239    5000.0
277091    5000.0
278569    5000.0
281640    5000.0
282200    5000.0
313828    5000.0
313959    5000.0
424072    5000.0
466181    5000.0
467343    5000.0
490720    5000.0
490729    5000.0
491042    5000.0
Name: property_value, dtype: float64

In [110]:
df["property_value"].nlargest(20)

84745     2.147484e+09
989384    2.147484e+09
478372    1.581075e+09
478371    1.449595e+09
510969    9.999950e+08
944186    9.590050e+08
505949    9.000050e+08
496068    8.750050e+08
494024    8.190050e+08
83353     8.000050e+08
500909    8.000050e+08
24286     6.520850e+08
494564    6.500050e+08
508947    6.000050e+08
497164    5.800050e+08
936358    5.500050e+08
901727    5.489050e+08
374304    5.000050e+08
811120    3.608050e+08
901726    3.534750e+08
Name: property_value, dtype: float64

In [111]:
low_property_count = (
    df["property_value"] == 5000
).sum()

low_property_pct = (
    low_property_count
    / df["property_value"].notna().sum()
    * 100
)

low_property_count, low_property_pct

(np.int64(114), np.float64(0.014228923407203081))

In [112]:
for threshold in [
    10_000_000,
    25_000_000,
    50_000_000,
    100_000_000,
    500_000_000,
    1_000_000_000
]:
    count = (df["property_value"] > threshold).sum()
    pct = (
        count
        / df["property_value"].notna().sum()
        * 100
    )

    print(threshold, count, round(pct, 4))

10000000 1715 0.2141
25000000 494 0.0617
50000000 229 0.0286
100000000 107 0.0134
500000000 18 0.0022
1000000000 4 0.0005


In [113]:
df.nlargest(20, "property_value")[
    [
        "property_value",
        "loan_amount",
        "loan_to_value_ratio",
        "action_label",
        "loan_type_label",
        "loan_purpose_label",
        "occupancy_label",
        "lei"
    ]
]

,property_value,loan_amount,loan_to_value_ratio,action_label,loan_type_label,loan_purpose_label,occupancy_label,lei
84745,2.147484e+09,1.650000e+05,0.008,Denied,Conventional,Other Purpose,Investment Property,3Y4U8VZURTYWI1W2K376
989384,2.147484e+09,2.650005e+09,66.300,Loan Originated,Conventional,Home Purchase,Investment Property,KB1H1DSPRFMYMCUFXT09
478372,1.581075e+09,5.982500e+07,51.480,Loan Originated,Conventional,Refinancing,Investment Property,549300DH8EI64ITBY388
478371,1.449595e+09,8.735500e+07,54.970,Loan Originated,Conventional,Refinancing,Investment Property,549300DH8EI64ITBY388
510969,9.999950e+08,3.550000e+05,0.010,Denied,Conventional,Home Purchase,Principal Residence,6BYL5QZYBDK8S7L73M02
944186,9.590050e+08,5.500000e+04,0.066,Denied,Conventional,Home Improvement,Principal Residence,X05BVSK68TQ7YTOSNR22
505949,9.000050e+08,8.500000e+04,0.010,Denied,Conventional,Home Improvement,Principal Residence,6BYL5QZYBDK8S7L73M02
496068,8.750050e+08,2.550000e+05,0.062,Denied,Conventional,Home Improvement,Principal Residence,6BYL5QZYBDK8S7L73M02
494024,8.190050e+08,8.500000e+04,0.061,Denied,Conventional,Home Improvement,Principal Residence,6BYL5QZYBDK8S7L73M02
83353,8.000050e+08,5.500000e+04,0.006,Denied,Conventional,Other Purpose,Principal Residence,3Y4U8VZURTYWI1W2K376


Extreme property values were reviewed in conjunction with loan amount and LTV. High property values were not automatically treated as invalid because some corresponded to large-dollar transactions, while others exhibited unusual loan-to-property-value relationships. Reported values were preserved for downstream analysis.

### 5.7 Income

In [114]:
df["income"].describe(
    percentiles=[0.001, 0.01, 0.05, 0.95, 0.99, 0.999]
)

count    8.851620e+05
mean     1.441520e+03
std      1.063446e+06
min     -1.801000e+04
0.1%     0.000000e+00
1%       0.000000e+00
5%       3.600000e+01
95%      5.630000e+02
99%      1.402000e+03
99.9%    6.000000e+03
max      1.000000e+09
Name: income, dtype: float64

In [115]:
df["income"].nsmallest(20)

743056   -18010.0
303007   -17231.0
74070    -11674.0
81580    -11312.0
525972   -11303.0
83163    -11132.0
975997    -8789.0
975671    -7920.0
368226    -5798.0
489878    -5601.0
375575    -5066.0
975674    -4910.0
975677    -4910.0
975680    -4910.0
514860    -4737.0
966072    -4111.0
84030     -4069.0
934226    -3886.0
984170    -3670.0
958533    -3657.0
Name: income, dtype: float64

In [116]:
df["income"].nlargest(20)

45552     999999999.0
374304     25553583.0
662281     11880258.0
284370     10800364.0
499839      8127644.0
903128      6964107.0
374179      3481430.0
373969      1776346.0
459869      1300090.0
323309      1049223.0
512218      1000000.0
937873       563369.0
505759       400130.0
903121       274000.0
674424       252000.0
46608        240400.0
40803        200000.0
944652       192431.0
944111       163770.0
318441       160000.0
Name: income, dtype: float64

In [117]:
for threshold in [
    10_000,
    25_000,
    50_000,
    100_000,
    1_000_000
]:
    count = (df["income"] > threshold).sum()
    pct = count / df["income"].notna().sum() * 100

    print(threshold, count, round(pct, 4))

10000 333 0.0376
25000 101 0.0114
50000 52 0.0059
100000 33 0.0037
1000000 10 0.0011


In [118]:
high_income_mask = df["income"] > 10_000

df.loc[
    high_income_mask,
    [
        "income",
        "loan_amount",
        "property_value",
        "action_label",
        "loan_type_label",
        "loan_purpose_label",
        "occupancy_label",
        "lei"
    ]
].sort_values("income", ascending=False).head(30)

,income,loan_amount,property_value,action_label,loan_type_label,loan_purpose_label,occupancy_label,lei
45552,999999999.0,495000.0,NaN,Withdrawn,Conventional,Home Purchase,Principal Residence,B4TYDEB6GKMZO031MB27
374304,25553583.0,40005000.0,500005000.0,Denied,Conventional,Cash-out Refinancing,Principal Residence,549300G4ESSS6M0MEG51
662281,11880258.0,725000.0,NaN,Denied,VA,Cash-out Refinancing,Principal Residence,549300DD5QQUHO6PCH70
284370,10800364.0,128005000.0,160005000.0,Denied,Conventional,Refinancing,Investment Property,549300Z77WUYJM3QG591
499839,8127644.0,855000.0,96005000.0,Denied,Conventional,Home Purchase,Investment Property,6BYL5QZYBDK8S7L73M02
903128,6964107.0,14505000.0,6305000.0,Loan Originated,Conventional,Home Purchase,Investment Property,IFQSIUC9AGQV2NE8CN25
374179,3481430.0,20005000.0,22005000.0,Denied,Conventional,Cash-out Refinancing,Principal Residence,549300G4ESSS6M0MEG51
373969,1776346.0,55000.0,5705000.0,Denied,Conventional,Cash-out Refinancing,Second Residence,549300G4ESSS6M0MEG51
459869,1300090.0,75000.0,NaN,Denied,Conventional,Other Purpose,Principal Residence,5493006MA7WP1WL8U431
323309,1049223.0,155000.0,NaN,Withdrawn,Conventional,Other Purpose,Principal Residence,5493001GCBD5XGNIC815


In [119]:
df.loc[
    high_income_mask,
    "lei"
].value_counts().head(10)

lei
6BYL5QZYBDK8S7L73M02    28
5493001GCBD5XGNIC815    14
E57ODZWZ7FF32TWEFA76    14
7H6GLXDRUGQFU57RNE97    14
B4TYDEB6GKMZO031MB27    13
X05BVSK68TQ7YTOSNR22    13
549300HW662MN1WU8550    12
5493004AS1SPBQOFDR49    11
KB1H1DSPRFMYMCUFXT09    10
3Y4U8VZURTYWI1W2K376     9
Name: count, dtype: int64

A broader threshold of 10,000 (reported in thousands of dollars) was used for exploratory review, while the more conservative 100,000 threshold was selected for the final extreme-value flag.

In [120]:
df["income_extreme_flag"] = pd.NA

df.loc[
    df["income"].notna(),
    "income_extreme_flag"
] = (
    df.loc[
        df["income"].notna(),
        "income"
    ] > 100_000
)

df["income_extreme_flag"] = (
    df["income_extreme_flag"].astype("boolean")
)

In [121]:
df["income_extreme_flag"].value_counts(dropna=False)

income_extreme_flag
False    885129
<NA>     140957
True         33
Name: count, dtype: Int64

## 6. Post-Cleaning Validation

Confirm that cleaned variables have the intended data types, categorical mappings are complete, derived fields reconcile with the source data, and no rows were unintentionally lost during cleaning.

In [122]:
df.shape

(1026119, 38)

In [123]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1026119 entries, 0 to 1026118
Data columns (total 38 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   activity_year             1026119 non-null  int64  
 1   lei                       1026119 non-null  str    
 2   state_code                1026119 non-null  str    
 3   county_code               1019645 non-null  float64
 4   census_tract              1017653 non-null  float64
 5   action_taken              1026119 non-null  int64  
 6   loan_type                 1026119 non-null  int64  
 7   loan_purpose              1026119 non-null  int64  
 8   loan_amount               1026119 non-null  float64
 9   loan_to_value_ratio       688449 non-null   float64
 10  interest_rate             642397 non-null   float64
 11  rate_spread               483636 non-null   float64
 12  total_loan_costs          435891 non-null   float64
 13  loan_term                 1000210 non-

In [124]:
validation_cols = [
    "interest_rate",
    "loan_to_value_ratio",
    "rate_spread",
    "total_loan_costs",
    "loan_term",
    "property_value",
    "income",
    "dti_numeric",
    "dti_category",
    "loan_type_label",
    "loan_purpose_label",
    "action_label",
    "occupancy_label",
    "ltv_extreme_flag",
    "rate_spread_extreme_flag",
    "loan_cost_extreme_flag",
    "loan_term_extreme_flag",
    "income_extreme_flag"
]

validation_summary = pd.DataFrame({
    "dtype": df[validation_cols].dtypes.astype(str),
    "non_missing": df[validation_cols].notna().sum(),
    "missing": df[validation_cols].isna().sum()
})

validation_summary

,dtype,non_missing,missing
interest_rate,float64,642397,383722
loan_to_value_ratio,float64,688449,337670
rate_spread,float64,483636,542483
total_loan_costs,float64,435891,590228
loan_term,float64,1000210,25909
property_value,float64,801185,224934
income,float64,885162,140957
dti_numeric,float64,357791,668328
dti_category,object,659408,366711
loan_type_label,str,1026119,0


### 6.1 Validation Assertions

In [125]:
assert len(df) == 1_026_119

assert df["loan_type_label"].isna().sum() == 0
assert df["loan_purpose_label"].isna().sum() == 0
assert df["action_label"].isna().sum() == 0
assert df["occupancy_label"].isna().sum() == 0

assert df["ltv_extreme_flag"].notna().sum() == df["loan_to_value_ratio"].notna().sum()
assert df["rate_spread_extreme_flag"].notna().sum() == df["rate_spread"].notna().sum()
assert df["loan_cost_extreme_flag"].notna().sum() == df["loan_cost_pct"].notna().sum()
assert df["loan_term_extreme_flag"].notna().sum() == df["loan_term"].notna().sum()
assert df["income_extreme_flag"].notna().sum() == df["income"].notna().sum()

In [126]:
assert (
    df["dti_category"].isna().sum()
    ==
    df["debt_to_income_ratio"].isna().sum()
    + (df["debt_to_income_ratio"] == "Exempt").sum()
)

## 7. Cleaning Summary and Data Quality Findings

The cleaning process preserved all 1,026,119 original records while improving the analytical representation of selected HMDA variables.

Categorical codes for loan type, loan purpose, application action, occupancy type, and denial reasons were mapped to descriptive labels while retaining the original coded fields for traceability.

Several variables originally stored as strings were converted to numeric form. Non-numeric `"Exempt"` values were treated as missing for numerical analysis rather than assigned artificial numeric values.

Debt-to-income ratio required separate treatment because the source data contains both exact numerical values and reported ranges. Exact values were preserved in `dti_numeric`, while range-based observations were retained in `dti_category` to avoid introducing artificial precision.

Extreme numerical observations were investigated rather than automatically removed. Reported values were preserved, while review flags were created for unusually high loan-to-value ratios, rate spreads, loan-cost ratios, loan terms, and reported income. These flags indicate observations requiring analytical caution and should not be interpreted as confirmed data errors.

Post-cleaning validation confirmed that no rows were unintentionally removed and that the major categorical mappings, derived variables, and quality flags reconcile with their source fields.

### Key Data Quality Findings

- Extremely high loan-to-value ratios are rare and were retained with an analytical review flag.
- Extreme rate-spread observations are highly concentrated in a small number of reporting institutions, including one LEI responsible for most values above 100.
- Very high loan-cost-to-loan-amount ratios are uncommon and frequently occur among purchased loans and very small reported loan amounts.
- Loan terms above 480 months are extremely rare and highly concentrated within one reporting institution.
- Extreme property values were reviewed alongside loan amounts and LTV and were not automatically classified as invalid.
- Reported income contains a small number of extremely large values; these were preserved and flagged rather than removed.

## 8. Save Cleaned Dataset

In [127]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

In [128]:
output_path = processed_dir / "hmda_ca_2024_cleaned.parquet"

df.to_parquet(output_path, index=False)

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

In [ ]:
output_path.exists(), output_path